# Lab 5.1 – Wie eine Maschine aus Daten lernt

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

**Frage:** Kauft ein Kunde in den drei Monaten nach dem Stichtag 30.06.2026 wieder bei Nordwind?

| Begriff | Hier |
| --- | --- |
| Merkmale (Features) | RFM je Kunde bis zum Stichtag: `tage_seit_kauf`, `haeufigkeit`, `umsatz` |
| Zielgröße | `wiederkauf`: 1 = kauft von Juli bis September 2026 wieder, 0 = nicht |
| Modell | logistische Regression (`LogisticRegression`) |
| Messlatte | „alle kaufen wieder“ – so oft liegt man ohne Modell richtig |

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
df = pd.read_csv(EINGABE, parse_dates=["datum"])
print(len(df), "Buchungen von", df["monat"].min(), "bis", df["monat"].max())

## Schritt 1 (fertig): Merkmale und Zielgröße je Kunde

Die Merkmale stammen nur aus Buchungen **bis** zum Stichtag (RFM wie in Lab 2.3). Die Zielgröße steht in den drei Monaten **danach** – das Modell soll aus der Vergangenheit auf die Zukunft schließen.

In [ ]:
STICHTAG = pd.Timestamp("2026-06-30")
ENDE = pd.Timestamp("2026-09-30")              # drei Monate nach dem Stichtag

# Merkmale: RFM je Kunde, nur aus Buchungen bis zum Stichtag
bis = df[df["datum"] <= STICHTAG]
kunden = bis.groupby("kunde_id").agg(
    letzter_kauf=("datum", "max"),
    haeufigkeit=("beleg_nr", "count"),
    umsatz=("umsatz", "sum"),
)
kunden["tage_seit_kauf"] = (STICHTAG - kunden["letzter_kauf"]).dt.days

# Zielgröße: kauft der Kunde zwischen Stichtag und ENDE wieder? 1 = ja, 0 = nein
danach = df[(df["datum"] > STICHTAG) & (df["datum"] <= ENDE)]
kunden["wiederkauf"] = kunden.index.isin(danach["kunde_id"]).astype(int)

kunden = kunden[["tage_seit_kauf", "haeufigkeit", "umsatz", "wiederkauf"]].round(2)
print(len(kunden), "Kunden, davon kaufen wieder:", kunden["wiederkauf"].sum())
kunden.head()

## Schritt 2: Trainings- und Testdaten trennen

`train_test_split` legt einen Teil der Kunden zur Seite (Testdaten). Das Modell lernt nur aus dem Rest (Trainingsdaten) und wird danach an den Testdaten gemessen. `random_state=42` sorgt dafür, dass bei jedem Lauf dieselben Kunden im Test landen.

In [ ]:
# >>> AUFGABE 1: Testanteil ändern (erst 0.3, dann 0.5) – siehe Übung, Aufgabe 3
TESTANTEIL = 0.2
# <<< ENDE AUFGABE 1

X = kunden[["tage_seit_kauf", "haeufigkeit", "umsatz"]]   # Merkmale (Features)
y = kunden["wiederkauf"]                                    # Zielgröße
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TESTANTEIL, random_state=42)
print("Training:", len(X_train), "Kunden · Test:", len(X_test), "Kunden")

## Schritt 3 (fertig): fit → predict → bewerten

Genauigkeit = Anteil der Testkunden, die das Modell richtig einordnet. Die Messlatte „alle kaufen wieder“ liegt richtig bei genau den Kunden, die wirklich wieder kaufen.

In [ ]:
modell = LogisticRegression(max_iter=1000)
modell.fit(X_train, y_train)                   # lernen: nur aus den Trainingsdaten
vorhersage = modell.predict(X_test)            # vorhersagen: für Kunden, die das Modell nie gesehen hat

genauigkeit = accuracy_score(y_test, vorhersage)
messlatte = y_test.mean()                      # Anteil der Testkunden mit wiederkauf = 1
print(f"Testanteil:                          {TESTANTEIL:.0%} ({len(X_test)} Kunden)")
print(f"Genauigkeit Modell (Testdaten):      {genauigkeit:.1%}")
print(f"Messlatte 'alle kaufen wieder':      {messlatte:.1%}")
print(f"Genauigkeit Modell (Trainingsdaten): {modell.score(X_train, y_train):.1%}")

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_modell():
    if not 0.1 <= TESTANTEIL <= 0.5:
        return f"nicht OK – TESTANTEIL = {TESTANTEIL}: einen Wert zwischen 0.1 und 0.5 verwenden."
    if genauigkeit <= messlatte:
        return f"nicht OK – Modell ({genauigkeit:.1%}) schlägt die Messlatte ({messlatte:.1%}) nicht."
    return f"OK – Testanteil {TESTANTEIL:.0%}: Modell {genauigkeit:.1%} liegt über der Messlatte {messlatte:.1%}"


print(pruefe_modell())

## Schritt 4 (fertig): Grenzfälle – wo das Modell unsicher ist

`predict_proba` liefert je Kunde die geschätzte Wahrscheinlichkeit für einen Wiederkauf (0 bis 1); ab 0,5 sagt das Modell 1 voraus. Kunden zwischen 0,2 und 0,8 sind Grenzfälle: Kandidaten für einen Anruf des Vertriebs.

In [ ]:
ergebnis = X_test.copy()
ergebnis["wiederkauf"] = y_test
ergebnis["vorhersage"] = vorhersage
ergebnis["wahrscheinlichkeit"] = modell.predict_proba(X_test)[:, 1].round(2)   # Chance auf Wiederkauf
grenzfaelle = ergebnis[ergebnis["wahrscheinlichkeit"].between(0.2, 0.8)]        # Modell ist unsicher
grenzfaelle.sort_values("wahrscheinlichkeit")

## Erweiterung: Verwechslungsmatrix

Welche Fehler macht das Modell? Prompt und Lösung: Übung, Erweiterung.

In [ ]:
# Erweiterung: Verwechslungsmatrix (siehe Übung)
